In [ ]:
# ---- SETTINGS: you can leave everything as it is and click "Run all" ----

INCLUDE_WEATHER = True     # True = also build part 2 (real weather for the store's 17 cities). False = orders only.
ORDERS_PER_MINUTE = 20     # about 20 orders a minute
MUTE_REGION = None         # to stage an outage, write a region: "North America", "Europe", "Asia Pacific" or "Latin America"
RUN_MINUTES = 120          # the store stops by itself after this many minutes

In [ ]:
# ---- SETUP: builds everything in THIS workspace. Safe to run again: whatever already exists is left as it is. ----
import base64, hashlib, hmac, json, math, random, time, uuid, urllib.error, urllib.parse, urllib.request
from datetime import datetime, timedelta, timezone
import notebookutils

WORKSPACE = notebookutils.runtime.context["currentWorkspaceId"]
API = "https://api.fabric.microsoft.com/v1/"
FABRIC = API + f"workspaces/{WORKSPACE}/"
EVENTHOUSE, ORDERS_TABLE, WEATHER_TABLE = "AuroraLive", "OrdersLive", "WeatherLive"
ORDERS_STREAM, WEATHER_STREAM = "AuroraOrders", "AuroraWeather"
DASHBOARD, AGENT, ACTIVATOR = "Aurora Live Orders", "Aurora Live Agent", "AuroraOrdersWatch"


def call(method, url, body=None, audience="pbi"):
    """One REST call with the identity of whoever runs this notebook. Waits for long-running operations."""
    data = None if body is None else json.dumps(body).encode("utf-8")
    request = urllib.request.Request(url, data=data, method=method)
    request.add_header("Authorization", "Bearer " + notebookutils.credentials.getToken(audience))
    if data is not None:
        request.add_header("Content-Type", "application/json")
    try:
        with urllib.request.urlopen(request, timeout=180) as response:
            status, headers, raw = response.status, response.headers, response.read()
    except urllib.error.HTTPError as error:
        raise RuntimeError(f"{method} {url.split('/v1/')[-1][:70]} failed ({error.code}): {error.read()[:400].decode('utf-8', 'replace')}")
    if status == 202 and headers.get("x-ms-operation-id"):
        operation = API + "operations/" + headers["x-ms-operation-id"]
        for _ in range(100):
            time.sleep(3)
            state = call("GET", operation)
            if state.get("status") == "Succeeded":
                try:
                    return call("GET", operation + "/result")
                except RuntimeError:
                    return {}
            if state.get("status") == "Failed":
                raise RuntimeError(f"{method} {url.split('/v1/')[-1][:70]} failed: {json.dumps(state)[:400]}")
        raise RuntimeError("an operation took too long: run the cell again")
    return json.loads(raw) if raw.strip() else {}


def find(kind, name):
    return next((item for item in call("GET", FABRIC + kind).get("value", []) if item["displayName"] == name), None)


def part(path, content):
    text = content if isinstance(content, str) else json.dumps(content)
    return {"path": path, "payload": base64.b64encode(text.encode("utf-8")).decode(), "payloadType": "InlineBase64"}


def kusto(text):
    """Runs a KQL query or command on the AuroraLive database. Returns the rows."""
    url = QUERY_URI + ("/v1/rest/mgmt" if text.lstrip().startswith(".") else "/v1/rest/query")
    table = call("POST", url, {"db": EVENTHOUSE, "csl": text}, audience=QUERY_URI)["Tables"][0]
    return table["Rows"]


def done(text):
    print("  [ok]", text)


# ---------------------------------------------------------------- 1. the Eventhouse and its tables
print("1. Eventhouse and tables")
if not find("eventhouses", EVENTHOUSE):
    call("POST", FABRIC + "eventhouses", {"displayName": EVENTHOUSE, "description": "Live orders of the Aurora demo store (simulated)"})
database = None
for _ in range(40):
    database = find("kqlDatabases", EVENTHOUSE)
    if database and database.get("properties", {}).get("queryServiceUri"):
        break
    time.sleep(5)
if not database:
    raise RuntimeError("The KQL database is not ready yet. Wait a minute and run this cell again.")
QUERY_URI = database["properties"]["queryServiceUri"]
done(f"Eventhouse {EVENTHOUSE}")
kusto(""".create-merge table OrdersLive (
    OrderId: string, OrderTime: datetime, ProductId: long, Product: string,
    Category: string, RegionId: long, Region: string, City: string,
    Quantity: long, UnitPrice: real, SalesAmount: real,
    PaymentMethod: string, Channel: string)""")
done(f"table {ORDERS_TABLE}")
if INCLUDE_WEATHER:
    kusto(""".create-merge table WeatherLive (
    ReadingTime: datetime, City: string, Region: string, Latitude: real, Longitude: real,
    TempC: real, PrecipMm: real, RainRate: real, SnowCm: real, WindKmh: real,
    WeatherCode: long, Condition: string, IntervalMin: long, Source: string)""")
    done(f"table {WEATHER_TABLE}")

# ---------------------------------------------------------------- 2. the Eventstreams (the front doors)
print("2. Eventstreams")


def eventstream(name, source, table):
    if find("eventstreams", name):
        return done(f"Eventstream {name} (already there)")
    topology = {"compatibilityLevel": "1.0",
                "sources": [{"name": source, "type": "CustomEndpoint", "properties": {}}],
                "streams": [{"name": name + "-stream", "type": "DefaultStream", "properties": {}, "inputNodes": [{"name": source}]}],
                "operators": [],
                "destinations": [{"name": table + "ToEventhouse", "type": "Eventhouse",
                                  "properties": {"dataIngestionMode": "ProcessedIngestion", "workspaceId": WORKSPACE,
                                                 "itemId": database["id"], "databaseName": EVENTHOUSE, "tableName": table,
                                                 "inputSerialization": {"type": "Json", "properties": {"encoding": "UTF8"}}},
                                  "inputNodes": [{"name": name + "-stream"}]}]}
    call("POST", FABRIC + "items", {"displayName": name, "type": "Eventstream", "definition": {"parts": [
        part("eventstream.json", topology),
        part("eventstreamProperties.json", {"retentionTimeInDays": 1, "eventThroughputLevel": "Low"})]}})
    done(f"Eventstream {name}")


eventstream(ORDERS_STREAM, "StoreOrders", ORDERS_TABLE)
if INCLUDE_WEATHER:
    eventstream(WEATHER_STREAM, "WeatherFeed", WEATHER_TABLE)

# ---------------------------------------------------------------- 3. a month of order history
print("3. History")
if kusto(f"{ORDERS_TABLE} | count")[0][0] < 100000:
    kusto(""".set-or-append OrdersLive <|
let products = datatable(ProductId: long, Product: string, Category: string, UnitPrice: real, Popular: long) [
    1, "Summit 2P Tent", "Camping", 331, 94,          2, "Basecamp 6P Tent", "Camping", 667, 50,
    3, "Arctic Sleeping Bag", "Camping", 217, 70,     4, "Trailhead 45L Pack", "Hiking", 179, 120,
    5, "Ridge Trekking Poles", "Hiking", 85, 85,      6, "Alpine Hiking Boots", "Hiking", 208, 104,
    7, "Storm Shell Jacket", "Apparel", 247, 101,     8, "Thermal Base Layer", "Apparel", 75, 133,
    9, "Summit Fleece", "Apparel", 122, 114,          10, "Glacier Water Bottle", "Accessories", 33, 154,
    11, "Headlamp Pro", "Accessories", 56, 120,       12, "Trail First-Aid Kit", "Accessories", 43, 92];
let cities = datatable(City: string, Region: string, RegionId: long, Popular: long) [
    "Vancouver", "North America", 1, 78,  "Seattle", "North America", 1, 78,  "Denver", "North America", 1, 78,
    "Toronto", "North America", 1, 78,    "New York", "North America", 1, 78,
    "London", "Europe", 2, 54,   "Berlin", "Europe", 2, 54,   "Oslo", "Europe", 2, 54,
    "Madrid", "Europe", 2, 54,   "Zurich", "Europe", 2, 54,
    "Tokyo", "Asia Pacific", 3, 40,     "Sydney", "Asia Pacific", 3, 40,
    "Auckland", "Asia Pacific", 3, 40,  "Singapore", "Asia Pacific", 3, 40,
    "Santiago", "Latin America", 4, 30, "Bogota", "Latin America", 4, 30, "Sao Paulo", "Latin America", 4, 30];
// a product or a city appears in its list as many times as it is popular, so a random pick follows the popularity
let productPick = toscalar(products | mv-expand n = range(1, Popular) to typeof(long) | summarize make_list(ProductId));
let cityPick = toscalar(cities | mv-expand n = range(1, Popular) to typeof(long) | summarize make_list(City));
let payments = dynamic(["Card", "Card", "Card", "Card", "Card", "Card", "PayPal", "PayPal", "Apple Pay", "Bank transfer"]);
let quantities = dynamic([1, 1, 1, 1, 1, 1, 1, 2, 2, 3]);
let channels = dynamic(["Web", "Web", "App"]);
// how busy each hour of the day is (UTC)
let curve = dynamic([0.62, 0.55, 0.50, 0.48, 0.50, 0.56, 0.66, 0.78, 0.90, 1.00, 1.08, 1.14,
                     1.20, 1.26, 1.32, 1.38, 1.42, 1.45, 1.42, 1.34, 1.20, 1.02, 0.84, 0.70]);
range Minute from bin(ago(28d), 1m) to bin(now(), 1m) - 1m step 1m
| extend Orders = toint(round(20 * todouble(curve[hourofday(Minute)])
                  * iff(dayofweek(Minute) == 0d or dayofweek(Minute) == 6d, 1.12, 1.0) * (0.75 + 0.5 * rand())))
| where Orders > 0
| mv-expand i = range(1, Orders) to typeof(long)
| extend OrderTime = Minute + totimespan(tolong(rand() * 600000000)),
         ProductId = tolong(productPick[toint(rand() * array_length(productPick))]),
         City = tostring(cityPick[toint(rand() * array_length(cityPick))]),
         Quantity = tolong(quantities[toint(rand() * 10)])
| lookup kind=leftouter products on ProductId
| lookup kind=leftouter cities on City
| project OrderId = strcat("ORD-", tolong((OrderTime - datetime(1970-01-01)) / 1ms), "-", toint(rand() * 900) + 100),
          OrderTime, ProductId, Product, Category, RegionId, Region, City, Quantity, UnitPrice,
          SalesAmount = round(Quantity * UnitPrice, 2),
          PaymentMethod = tostring(payments[toint(rand() * 10)]), Channel = tostring(channels[toint(rand() * 3)])""")
    done(f"28 days of orders: {kusto(ORDERS_TABLE + ' | count')[0][0]:,} rows")
else:
    done("the history is already there")

# ---------------------------------------------------------------- 4. the dashboard
print("4. Dashboard")
if not find("items?type=KQLDashboard", DASHBOARD):
    source_id, page, queries, tiles = str(uuid.uuid4()), str(uuid.uuid4()), [], []

    def tile(title, visual, text, x, y, width, height):
        queries.append({"id": str(uuid.uuid4()), "dataSource": {"kind": "inline", "dataSourceId": source_id}, "text": text, "usedVariables": []})
        tiles.append({"id": str(uuid.uuid4()), "title": title, "visualType": visual, "pageId": page,
                      "layout": {"x": x, "y": y, "width": width, "height": height},
                      "queryRef": {"kind": "query", "queryId": queries[-1]["id"]}, "visualOptions": {}})

    tile("Orders in the last minute", "card", """OrdersLive
| where OrderTime > ago(1m)
| summarize ['Orders in the last minute'] = count()""", 0, 0, 5, 4)
    tile("Orders per minute by region", "line", """OrdersLive
| where OrderTime >= bin(ago(30m), 1m) and OrderTime < bin(now(), 1m)
| make-series Orders = count() default = 0 on OrderTime from bin(ago(30m), 1m) to bin(now(), 1m) step 1m by Region
| mv-expand OrderTime to typeof(datetime), Orders to typeof(long)
| project OrderTime, Region, Orders""", 5, 0, 19, 8)
    tile("The latest orders", "table", """OrdersLive
| top 8 by OrderTime desc
| project OrderTime, Product, City, Region, SalesAmount, PaymentMethod""", 0, 8, 24 if not INCLUDE_WEATHER else 12, 8)
    if INCLUDE_WEATHER:
        tile("Weather in the store's cities, right now", "map", """WeatherLive
| where IntervalMin == 15 and ReadingTime > ago(1h)
| summarize arg_max(ReadingTime, *) by City
| project Longitude, Latitude,
          Weather = iff(RainRate >= 0.4, "Raining now", iff(TempC < 5, "Cold", iff(TempC > 26, "Hot", "Mild"))),
          Size = iff(RainRate >= 0.4, 3.0, 1.0)""", 12, 8, 12, 8)
        tile("The weather readings", "table", """WeatherLive
| where IntervalMin == 15 and ReadingTime > ago(1h)
| summarize arg_max(ReadingTime, *) by City
| project City, ['Temp (C)'] = TempC, ['Rain (mm/h)'] = RainRate, Condition, ['As of'] = ReadingTime
| order by ['Rain (mm/h)'] desc, City asc""", 0, 16, 24, 7)
    call("POST", FABRIC + "items", {"displayName": DASHBOARD, "type": "KQLDashboard", "definition": {"parts": [part("RealTimeDashboard.json", {
        "$schema": "https://dataexplorer.azure.com/static/d/schema/52/dashboard.json", "schema_version": "52", "title": DASHBOARD,
        "autoRefresh": {"enabled": True, "defaultInterval": "10s", "minInterval": "10s"},
        "tiles": tiles, "baseQueries": [], "parameters": [], "queries": queries, "pages": [{"name": "Live", "id": page}],
        "dataSources": [{"id": source_id, "name": EVENTHOUSE, "clusterUri": QUERY_URI, "database": database["id"],
                         "kind": "kusto-trident", "scopeId": "kusto-trident", "workspace": WORKSPACE}]})]}})
    done(f"dashboard {DASHBOARD}")
else:
    done("the dashboard is already there")

# ---------------------------------------------------------------- 5. the rules (created STOPPED)
print("5. Rules in Activator")


def me():
    """The email of whoever runs this notebook: the rules send their Teams message there."""
    claims = notebookutils.credentials.getToken("pbi").split(".")[1]
    claims = json.loads(base64.urlsafe_b64decode(claims + "=" * (-len(claims) % 4)))
    return claims.get("upn") or claims.get("unique_name") or claims.get("preferred_username")


def arg(name, kind, value):
    return {"name": name, "type": kind, "value": value}


def ref(kind, entity, name=None, type_="complex"):
    r = {"kind": kind, "type": type_, "arguments": [arg("entityId", "string", entity)]}
    if name:
        r["name"] = name
    return r


def view(uid, name, kind, container, instance=None, parent=None, settings=None):
    payload = {"name": name, "parentContainer": {"targetUniqueIdentifier": container}, "definition": {"type": kind}}
    if parent:
        payload["parentObject"] = {"targetUniqueIdentifier": parent}
    if instance is not None:
        payload["definition"]["instance"] = json.dumps(instance, separators=(",", ":"))
    if settings:
        payload["definition"]["settings"] = settings
    return {"uniqueIdentifier": uid, "payload": payload, "type": "timeSeriesView-v1"}


def attribute(uid, name, event, field, kind, container, parent):
    return view(uid, name, "Attribute", container, {"templateId": "BasicEventAttribute", "templateVersion": "1.2.4", "steps": [
        {"name": "EventSelectStep", "id": str(uuid.uuid4()), "rows": [
            {"name": "EventSelector", "kind": "Event", "arguments": [ref("EventReference", event, "event")]},
            {"name": "EventFieldSelector", "kind": "EventField", "arguments": [arg("fieldName", "string", field)]}]},
        {"name": "EventComputeStep", "id": str(uuid.uuid4()), "rows": [
            {"name": "TypeAssertion", "kind": "TypeAssertion", "arguments": [arg("op", "string", kind), arg("format", "string", "")]}]}]},
                parent=parent)


def watch(container, title, query, key, number, when, threshold, rule_name, headline, message, recipient):
    """A query that runs every minute, one object per region or city, and a rule on a number of that object."""
    source, event, thing, identity, split, value, label, rule = (str(uuid.uuid4()) for _ in range(8))
    now = datetime.now(timezone.utc)
    stamp = "%Y-%m-%dT%H:%M:%SZ"
    return [
        {"uniqueIdentifier": source, "type": "kqlSource-v1", "payload": {
            "name": title, "runSettings": {"executionIntervalInSeconds": 60}, "query": {"queryString": query},
            "eventhouseItem": {"itemId": database["id"], "workspaceId": WORKSPACE, "itemType": "KustoDatabase"},
            "queryParameters": [{"name": "startTime", "type": "DURATION_START", "value": (now - timedelta(minutes=10)).strftime(stamp)},
                                {"name": "endTime", "type": "DURATION_END", "value": now.strftime(stamp)}],
            "eventTimeSettings": {"timeFieldName": "Minute", "ingestionDelayInSeconds": 20, "timeZone": "UTC"},
            "metadata": {"workspaceId": WORKSPACE, "measureName": "", "querySetId": "", "queryId": ""},
            "parentContainer": {"targetUniqueIdentifier": container}}},
        view(event, title + " (events)", "Event", container, {"templateId": "SourceEvent", "templateVersion": "1.2.4", "steps": [
            {"name": "SourceEventStep", "id": str(uuid.uuid4()), "rows": [
                {"name": "SourceSelector", "kind": "SourceReference", "arguments": [arg("entityId", "string", source)]}]}]}),
        view(thing, key, "Object", container),
        view(identity, key, "Attribute", container, {"templateId": "IdentityPartAttribute", "templateVersion": "1.2.4", "steps": [
            {"name": "IdPartStep", "id": str(uuid.uuid4()), "rows": [{"name": "TypeAssertion", "kind": "TypeAssertion", "arguments": [
                arg("op", "string", "Text"), arg("format", "string", "")]}]}]}, parent=thing),
        view(split, title + " per " + key.lower(), "Event", container, {"templateId": "SplitEvent", "templateVersion": "1.2.4", "steps": [
            {"name": "SplitEventStep", "id": str(uuid.uuid4()), "rows": [
                {"name": "EventSelector", "kind": "Event", "arguments": [ref("EventReference", event, "event")]},
                {"name": "FieldIdMapping", "kind": "FieldIdMapping", "arguments": [
                    arg("fieldName", "string", key), ref("AttributeReference", identity, "idPart")]},
                {"name": "SplitEventOptions", "kind": "EventOptions", "arguments": [arg("isAuthoritative", "boolean", True)]}]}]},
             parent=thing),
        attribute(value, number, split, number, "Number", container, thing),
        attribute(label, key + " name", split, key, "Text", container, thing),
        view(rule, rule_name, "Rule", container, {"templateId": "AttributeTrigger", "templateVersion": "1.2.4", "steps": [
            {"name": "ScalarSelectStep", "id": str(uuid.uuid4()), "rows": [
                {"name": "AttributeSelector", "kind": "Attribute", "arguments": [ref("AttributeReference", value, "attribute")]}]},
            {"name": "ScalarDetectStep", "id": str(uuid.uuid4()), "rows": [
                {"name": "NumberBecomes", "kind": "NumberBecomes", "arguments": [arg("op", "string", when), arg("value", "number", threshold)]}]},
            {"name": "ActStep", "id": str(uuid.uuid4()), "rows": [{"name": "TeamsBinding", "kind": "TeamsMessage", "arguments": [
                arg("messageLocale", "string", ""),
                {"name": "recipients", "type": "array", "values": [{"type": "string", "value": recipient}]},
                {"name": "headline", "type": "array", "values": [
                    {"name": "string", "type": "string", "value": headline + " "}, ref("AttributeReference", label),
                    {"name": "string", "type": "string", "value": " "}]},
                {"name": "optionalMessage", "type": "array", "values": [{"name": "string", "type": "string", "value": message}]},
                {"name": "additionalInformation", "type": "array", "values": [
                    {"kind": "NameReferencePair", "type": "complex", "arguments": [
                        arg("name", "string", number), ref("AttributeReference", value, "reference", "complexReference")]}]}]}]}]},
             parent=thing, settings={"shouldRun": False, "shouldApplyRuleOnUpdate": False})]


ORDERS_RULE_QUERY = """declare query_parameters(startTime:datetime, endTime:datetime);
let e = bin(endTime, 1m);
let s = bin(startTime, 1m) + 1m;
let regions = datatable(Region:string)["North America", "Europe", "Asia Pacific", "Latin America"];
let perMinute = OrdersLive
    | where OrderTime >= s - 10m and OrderTime < e
    | summarize c = count() by Region, M = bin(OrderTime, 1m);
range Minute from s to e step 1m
| extend k = 1
| join kind=inner (regions | extend k = 1) on k
| project Minute, Region
| join kind=leftouter perMinute on Region
| summarize OrdersLast10Minutes = sumif(c, M >= Minute - 10m and M < Minute) by Minute, Region
| order by Minute asc, Region asc"""
RAIN_RULE_QUERY = """declare query_parameters(startTime:datetime, endTime:datetime);
let e = bin(endTime, 1m);
let s = bin(startTime, 1m) + 1m;
let r = WeatherLive
    | where IntervalMin == 15 and ReadingTime > s - 31m and ReadingTime <= e
    | project City, ReadingTime, RainRate;
range Minute from s to e step 1m
| extend k = 1
| join kind=inner (r | extend k = 1) on k
| where ReadingTime <= Minute and ReadingTime > Minute - 30m
| summarize RainMax30 = max(RainRate) by Minute, City
| project Minute, City, WetInLast30Minutes = iff(RainMax30 >= 0.4, 1, 0)
| order by Minute asc, City asc"""
if not find("reflexes", ACTIVATOR):
    call("POST", FABRIC + "reflexes", {"displayName": ACTIVATOR, "description": "Rules on the live orders of the Aurora demo store"})
    container = str(uuid.uuid4())
    entities = [{"uniqueIdentifier": container, "payload": {"name": "Aurora rules", "type": "kqlQueries"}, "type": "container-v1"}]
    entities += watch(container, "Orders per region", ORDERS_RULE_QUERY, "Region", "OrdersLast10Minutes", "BecomesLessThan", 1.0,
                      "Region stopped ordering", "Orders stopped:", "No orders from this region in the last 10 minutes.", me())
    if INCLUDE_WEATHER:
        entities += watch(container, "Rain per city", RAIN_RULE_QUERY, "City", "WetInLast30Minutes", "BecomesGreaterThan", 0.0,
                          "Rain started in a city", "Rain started:", "A rainy reading just arrived for this city.", me())
    call("POST", FABRIC + f"reflexes/{find('reflexes', ACTIVATOR)['id']}/updateDefinition",
         {"definition": {"parts": [part("ReflexEntities.json", entities)]}})
    done(f"Activator {ACTIVATOR}: the rules exist and are STOPPED. Start them when you want to test (the guide shows where)")
else:
    done("the Activator item is already there")

# ---------------------------------------------------------------- 6. the Fabric data agent
print("6. Data agent")
ORDER_COLUMNS = [("OrderId", "String"), ("OrderTime", "DateTime"), ("ProductId", "Int64"), ("Product", "String"), ("Category", "String"),
                 ("RegionId", "Int64"), ("Region", "String"), ("City", "String"), ("Quantity", "Int64"), ("UnitPrice", "Double"),
                 ("SalesAmount", "Double"), ("PaymentMethod", "String"), ("Channel", "String")]
WEATHER_COLUMNS = [("ReadingTime", "DateTime"), ("City", "String"), ("Region", "String"), ("Latitude", "Double"), ("Longitude", "Double"),
                   ("TempC", "Double"), ("PrecipMm", "Double"), ("RainRate", "Double"), ("SnowCm", "Double"), ("WindKmh", "Double"),
                   ("WeatherCode", "Int64"), ("Condition", "String"), ("IntervalMin", "Int64"), ("Source", "String")]
ORDERS_ONLY = ("You answer questions about Aurora Outdoor's online store (a simulated demo store) from the table OrdersLive: one row per order "
               "(OrderTime, Product, Category, Region, City, Quantity, SalesAmount, PaymentMethod, Channel), about 28 days of history plus "
               "orders arriving right now. Every number in an answer must come from a query. Always state the time window you used. "
               "All times are UTC.")
WITH_WEATHER = """You answer questions about Aurora Outdoor's online store (a simulated demo store) from two tables in the AuroraLive KQL database. Every number in an answer must come from a query. Always state the time window you used. All times are UTC.

OrdersLive: one row per order (OrderTime, Product, Category, Region, City, Quantity, SalesAmount, PaymentMethod, Channel). It holds about 28 days of history plus orders arriving right now.

WeatherLive: REAL weather readings for the store's 17 cities (source: Open-Meteo). One row per city per reading.
- IntervalMin == 60: hourly history. ReadingTime is the END of the hour the reading covers.
- IntervalMin == 15: readings arriving now. Each covers the 15 minutes before ReadingTime.
- RainRate is millimetres per hour in both kinds of row. Raining means RainRate >= 0.4. Heavy rain means RainRate >= 2.
- Cold means TempC < 5. Hot means TempC > 26. Condition is the weather in words.

How to work with the weather:
- The weather in a city now: the latest reading, WeatherLive | where ReadingTime > ago(2h) | summarize arg_max(ReadingTime, *) by City.
- Weather and orders in the history: summarize OrdersLive by City and Hour = bin(OrderTime, 1h), and join to WeatherLive | where IntervalMin == 60 | extend Hour = ReadingTime - 1h, on City and Hour. Keep the hours with no orders of a product as 0 (start from the weather rows).
- Compare orders PER HOUR in rainy hours and in dry hours of the same city (or cold and not cold). Never compare totals: dry hours are far more numerous.
- "What does rain usually do in <city>": for the last 28 days give the average orders per hour of each product in rainy and in dry hours of that city, the ratio, and how many rainy hours that is based on. Name the products whose ratio stands out.
- "What to expect": the usual orders per hour in that weather, next to what the last 30 to 60 minutes actually show.
- The store's shoppers are simulated and were built to react to weather, so say what the data shows and do not present it as a discovery about real customers."""
try:
    if not find("dataAgents", AGENT):
        def table(name, columns):
            return {"id": str(uuid.uuid4()), "is_selected": True, "display_name": name, "type": "kusto.table", "description": None,
                    "children": [{"id": c, "is_selected": True, "display_name": c, "type": "kusto.column", "data_type": "System." + t,
                                  "description": None, "children": []} for c, t in columns]}

        def group(name, kind, children):
            return {"id": str(uuid.uuid4()), "is_selected": False, "display_name": name, "type": kind, "description": None, "children": children}

        schema = "https://developer.microsoft.com/json-schemas/fabric/item/dataAgent/definition/"
        source = {"$schema": schema + "dataSource/1.0.0/schema.json", "artifactId": database["id"], "workspaceId": WORKSPACE,
                  "dataSourceInstructions": None, "displayName": EVENTHOUSE, "type": "kusto", "userDescription": None, "metadata": None,
                  "elements": [group("Tables", "table_grouping", [table(ORDERS_TABLE, ORDER_COLUMNS)]
                                     + ([table(WEATHER_TABLE, WEATHER_COLUMNS)] if INCLUDE_WEATHER else [])),
                               group("Functions", "function_grouping", []), group("Shortcuts", "shortcut_grouping", []),
                               group("Materialized Views", "materialized_view_grouping", [])]}
        stage = {"$schema": schema + "stageConfiguration/1.0.0/schema.json", "aiInstructions": WITH_WEATHER if INCLUDE_WEATHER else ORDERS_ONLY}
        about = ("Answers questions about the live orders of the Aurora Outdoor online store (a simulated demo store)"
                 + (" and the real weather in its 17 cities" if INCLUDE_WEATHER else "") + ". All times are UTC.")
        call("POST", FABRIC + "dataAgents", {"displayName": AGENT, "description": about, "definition": {"parts": [
            part("Files/Config/data_agent.json", {"$schema": schema + "dataAgent/2.1.0/schema.json"}),
            part("Files/Config/draft/stage_config.json", stage),
            part(f"Files/Config/draft/kusto-{EVENTHOUSE}/datasource.json", source),
            part("Files/Config/publish_info.json", {"$schema": schema + "publishInfo/1.0.0/schema.json", "description": about}),
            part("Files/Config/published/stage_config.json", stage),
            part(f"Files/Config/published/kusto-{EVENTHOUSE}/datasource.json", source)]}})
        done(f"data agent {AGENT} (published)")
    else:
        done("the data agent is already there")
except RuntimeError as problem:
    print("  [!] The data agent could not be created from here:", str(problem)[:200])
    print("      Create it by hand: it is five clicks, step 6 of the guide.")

print()
print("Everything is in place. The next cell opens the store.")

In [ ]:
# ---- THE STORE: runs until RUN_MINUTES pass, or until you click "Interrupt kernel". Run it again whenever you want. ----
PRODUCTS = [  # id, name, category, price, how popular
    (1, "Summit 2P Tent", "Camping", 331, 94), (2, "Basecamp 6P Tent", "Camping", 667, 50),
    (3, "Arctic Sleeping Bag", "Camping", 217, 70), (4, "Trailhead 45L Pack", "Hiking", 179, 120),
    (5, "Ridge Trekking Poles", "Hiking", 85, 85), (6, "Alpine Hiking Boots", "Hiking", 208, 104),
    (7, "Storm Shell Jacket", "Apparel", 247, 101), (8, "Thermal Base Layer", "Apparel", 75, 133),
    (9, "Summit Fleece", "Apparel", 122, 114), (10, "Glacier Water Bottle", "Accessories", 33, 154),
    (11, "Headlamp Pro", "Accessories", 56, 120), (12, "Trail First-Aid Kit", "Accessories", 43, 92)]
REGIONS = {"North America": (1, 39), "Europe": (2, 27), "Asia Pacific": (3, 16), "Latin America": (4, 9)}   # id, share of orders
CITIES = {  # city: (latitude, longitude, region)
    "Vancouver": (49.28, -123.12, "North America"), "Seattle": (47.61, -122.33, "North America"),
    "Denver": (39.74, -104.99, "North America"), "Toronto": (43.65, -79.38, "North America"),
    "New York": (40.71, -74.01, "North America"), "London": (51.51, -0.13, "Europe"),
    "Berlin": (52.52, 13.41, "Europe"), "Oslo": (59.91, 10.75, "Europe"), "Madrid": (40.42, -3.70, "Europe"),
    "Zurich": (47.38, 8.54, "Europe"), "Tokyo": (35.68, 139.69, "Asia Pacific"),
    "Sydney": (-33.87, 151.21, "Asia Pacific"), "Auckland": (-36.85, 174.76, "Asia Pacific"),
    "Singapore": (1.35, 103.82, "Asia Pacific"), "Santiago": (-33.45, -70.67, "Latin America"),
    "Bogota": (4.71, -74.07, "Latin America"), "Sao Paulo": (-23.55, -46.63, "Latin America")}
CONDITIONS = {0: "Clear", 1: "Mostly clear", 2: "Partly cloudy", 3: "Overcast", 45: "Fog", 48: "Fog", 51: "Light drizzle",
              53: "Drizzle", 55: "Heavy drizzle", 61: "Light rain", 63: "Rain", 65: "Heavy rain", 71: "Light snow",
              73: "Snow", 75: "Heavy snow", 80: "Light showers", 81: "Showers", 82: "Heavy showers", 95: "Thunderstorm"}
PAYMENTS = [("Card", 62), ("PayPal", 20), ("Apple Pay", 12), ("Bank transfer", 6)]
QUANTITIES = [(1, 70), (2, 22), (3, 8)]
CHANNELS = [("Web", 64), ("App", 36)]
CURVE = [0.62, 0.55, 0.50, 0.48, 0.50, 0.56, 0.66, 0.78, 0.90, 1.00, 1.08, 1.14,
         1.20, 1.26, 1.32, 1.38, 1.42, 1.45, 1.42, 1.34, 1.20, 1.02, 0.84, 0.70]      # how busy each hour is (UTC)
# What the SIMULATED shoppers do with the REAL weather in their city:
RAIN, HEAVY_RAIN, COLD, HOT = 0.4, 2.0, 5.0, 26.0             # mm per hour, mm per hour, degrees C, degrees C
EFFECTS = {"rain": (["Storm Shell Jacket"], 3.0), "heavy rain": (["Storm Shell Jacket"], 4.0),
           "cold": (["Thermal Base Layer", "Summit Fleece"], 1.8), "hot": (["Glacier Water Bottle"], 2.2)}
CITY_NAMES = {region: [c for c, v in CITIES.items() if v[2] == region] for region in REGIONS}
TOTAL_POPULAR = sum(p[4] for p in PRODUCTS)
TOTAL_SHARE = sum(r[1] for r in REGIONS.values())


def endpoint(stream):
    """The address and key of an Eventstream's custom endpoint, read from Fabric (nothing to paste)."""
    item = find("eventstreams", stream)
    for _ in range(40):
        try:
            sources = call("GET", FABRIC + f"eventstreams/{item['id']}/topology").get("sources", [])
            custom = next(s for s in sources if s["type"] == "CustomEndpoint")
            keys = call("GET", FABRIC + f"eventstreams/{item['id']}/sources/{custom['id']}/connection")
            conn = dict(p.split("=", 1) for p in keys["accessKeys"]["primaryConnectionString"].split(";") if "=" in p)
            conn["url"] = "https://" + conn["Endpoint"].replace("sb://", "").strip("/") + "/" + conn["EntityPath"]
            return conn
        except (RuntimeError, StopIteration, KeyError):
            time.sleep(6)                                      # a new Eventstream needs a minute to wake up
    raise RuntimeError(f"The Eventstream {stream} is not ready yet. Wait a minute and run this cell again.")


def send(conn, events):
    uri = urllib.parse.quote_plus(conn["url"])
    expiry = str(int(time.time()) + 3600)
    signature = base64.b64encode(hmac.new(conn["SharedAccessKey"].encode(), f"{uri}\n{expiry}".encode(), hashlib.sha256).digest()).decode()
    token = f"SharedAccessSignature sr={uri}&sig={urllib.parse.quote_plus(signature)}&se={expiry}&skn={conn['SharedAccessKeyName']}"
    body = json.dumps([{"Body": json.dumps(e)} for e in events]).encode("utf-8")
    for attempt in range(5):
        request = urllib.request.Request(conn["url"] + "/messages?timeout=60&api-version=2014-01", data=body, method="POST")
        request.add_header("Authorization", token)
        request.add_header("Content-Type", "application/vnd.microsoft.servicebus.json")
        try:
            with urllib.request.urlopen(request, timeout=30) as response:
                return response.status
        except (urllib.error.URLError, TimeoutError):
            time.sleep(5)
    raise RuntimeError("The Eventstream did not accept the events. Wait a minute and run this cell again.")


def pick(pairs):
    return random.choices([p[0] for p in pairs], weights=[p[1] for p in pairs])[0]


def order(product=None, city=None):
    pid, name, category, price, _ = product or random.choices(PRODUCTS, weights=[p[4] for p in PRODUCTS])[0]
    if city is None:
        region = random.choices(list(REGIONS), weights=[r[1] for r in REGIONS.values()])[0]
        city = random.choice(CITY_NAMES[region])
    region = CITIES[city][2]
    quantity = pick(QUANTITIES)
    now = datetime.now(timezone.utc)
    return {"OrderId": f"ORD-{int(now.timestamp() * 1000)}-{random.randint(100, 999)}",
            "OrderTime": now.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z",
            "ProductId": pid, "Product": name, "Category": category, "RegionId": REGIONS[region][0], "Region": region,
            "City": city, "Quantity": quantity, "UnitPrice": price, "SalesAmount": quantity * price,
            "PaymentMethod": pick(PAYMENTS), "Channel": pick(CHANNELS)}


def weather_service(parameters):
    """Weather data by Open-Meteo.com (CC BY 4.0, free for non-commercial use, no key)."""
    latitudes = ",".join(str(c[0]) for c in CITIES.values())
    longitudes = ",".join(str(c[1]) for c in CITIES.values())
    url = f"https://api.open-meteo.com/v1/forecast?latitude={latitudes}&longitude={longitudes}&timezone=UTC&{parameters}"
    with urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": "fabric-demo"}), timeout=60) as response:
        return json.loads(response.read())


def reading(city, time_text, temp, precipitation, snow, wind, code, minutes):
    latitude, longitude, region = CITIES[city]
    precipitation = precipitation or 0.0
    return {"ReadingTime": time_text + ":00Z", "City": city, "Region": region, "Latitude": latitude, "Longitude": longitude,
            "TempC": temp, "PrecipMm": precipitation, "RainRate": round(precipitation * 60 / minutes, 2),   # millimetres per hour
            "SnowCm": snow or 0.0, "WindKmh": wind, "WeatherCode": code, "Condition": CONDITIONS.get(code, str(code)),
            "IntervalMin": minutes, "Source": "open-meteo.com"}


def weather_now():
    data = weather_service("current=temperature_2m,precipitation,snowfall,weather_code,wind_speed_10m")
    stamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M")
    readings = []
    for city, place in zip(CITIES, data):
        c = place["current"]
        readings.append(reading(city, stamp, c["temperature_2m"], c["precipitation"], c["snowfall"], c["wind_speed_10m"],
                                c["weather_code"], int(c["interval"] // 60)))
    return readings


def what_weather_does(r):
    tags = ["heavy rain"] if r["RainRate"] >= HEAVY_RAIN else ["rain"] if r["RainRate"] >= RAIN else []
    if r["TempC"] is not None and r["TempC"] < COLD:
        tags.append("cold")
    if r["TempC"] is not None and r["TempC"] > HOT:
        tags.append("hot")
    return tags


def how_many(expected):                     # a random whole number around "expected"
    limit, count, p = math.exp(-expected), 0, 1.0
    while expected > 0:
        p *= random.random()
        if p <= limit:
            break
        count += 1
    return count


orders = endpoint(ORDERS_STREAM)
weather = endpoint(WEATHER_STREAM) if INCLUDE_WEATHER else None

if INCLUDE_WEATHER and kusto(f"{WEATHER_TABLE} | where IntervalMin == 60 | count")[0][0] == 0:
    print("First run: sending 28 days of real weather history...")
    data = weather_service("hourly=temperature_2m,precipitation,snowfall,weather_code,wind_speed_10m&past_days=28&forecast_days=1")
    this_hour = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:00")
    history = []
    for city, place in zip(CITIES, data):
        h = place["hourly"]
        for i, t in enumerate(h["time"]):
            if t <= this_hour and h["temperature_2m"][i] is not None:
                history.append(reading(city, t, h["temperature_2m"][i], h["precipitation"][i], h["snowfall"][i],
                                       h["wind_speed_10m"][i], h["weather_code"][i], 60))
    for start in range(0, len(history), 300):
        send(weather, history[start:start + 300])
    for _ in range(60):                                        # wait until the database has all of it
        if kusto(f"{WEATHER_TABLE} | where IntervalMin == 60 | count")[0][0] >= len(history) * 0.98:
            break
        time.sleep(5)
    kusto(""".set-or-append OrdersLive <|
// What the SIMULATED shoppers did with the REAL weather of the last 28 days:
// rain = rain jackets x 3 (heavy rain x 4), cold = warm layers x 1.8, hot = water bottles x 2.2.
let effects = datatable(Weather: string, Product: string, ProductId: long, Category: string, UnitPrice: real, ExtraPerOrder: real) [
    "rain", "Storm Shell Jacket", 7, "Apparel", 247, 0.1633,
    "heavy rain", "Storm Shell Jacket", 7, "Apparel", 247, 0.2449,
    "cold", "Thermal Base Layer", 8, "Apparel", 75, 0.0860,
    "cold", "Summit Fleece", 9, "Apparel", 122, 0.0737,
    "hot", "Glacier Water Bottle", 10, "Accessories", 33, 0.1494];
let cities = datatable(City: string, RegionId: long, CityShare: real) [
    "Vancouver", 1, 0.0857,  "Seattle", 1, 0.0857,  "Denver", 1, 0.0857,  "Toronto", 1, 0.0857,  "New York", 1, 0.0857,
    "London", 2, 0.0593,     "Berlin", 2, 0.0593,   "Oslo", 2, 0.0593,    "Madrid", 2, 0.0593,   "Zurich", 2, 0.0593,
    "Tokyo", 3, 0.0440,      "Sydney", 3, 0.0440,   "Auckland", 3, 0.0440, "Singapore", 3, 0.0440,
    "Santiago", 4, 0.0330,   "Bogota", 4, 0.0330,   "Sao Paulo", 4, 0.0330];
let payments = dynamic(["Card", "Card", "Card", "Card", "Card", "Card", "PayPal", "PayPal", "Apple Pay", "Bank transfer"]);
let quantities = dynamic([1, 1, 1, 1, 1, 1, 1, 2, 2, 3]);
let channels = dynamic(["Web", "Web", "App"]);
let curve = dynamic([0.62, 0.55, 0.50, 0.48, 0.50, 0.56, 0.66, 0.78, 0.90, 1.00, 1.08, 1.14,
                     1.20, 1.26, 1.32, 1.38, 1.42, 1.45, 1.42, 1.34, 1.20, 1.02, 0.84, 0.70]);
WeatherLive
| where IntervalMin == 60 and ReadingTime > ago(28d) and ReadingTime <= now()
| extend HourStart = ReadingTime - 1h
| extend Weather = pack_array(iff(RainRate >= 2.0, "heavy rain", iff(RainRate >= 0.4, "rain", "")),
                              iff(TempC < 5, "cold", ""), iff(TempC > 26, "hot", ""))
| mv-expand Weather to typeof(string)
| where Weather != ""
| lookup kind=inner effects on Weather
| lookup kind=inner cities on City
| extend BaseOrders = 20 * 60 * todouble(curve[hourofday(HourStart)])
                      * iff(dayofweek(HourStart) == 0d or dayofweek(HourStart) == 6d, 1.12, 1.0) * CityShare
| extend Extra = toint(round(BaseOrders * ExtraPerOrder * (0.75 + 0.5 * rand())))
| where Extra > 0
| mv-expand i = range(1, Extra) to typeof(long)
| extend OrderTime = HourStart + totimespan(tolong(rand() * 36000000000)), Quantity = tolong(quantities[toint(rand() * 10)])
| project OrderId = strcat("ORD-", tolong((OrderTime - datetime(1970-01-01)) / 1ms), "-", toint(rand() * 900) + 100),
          OrderTime, ProductId, Product, Category, RegionId, Region, City, Quantity, UnitPrice,
          SalesAmount = round(Quantity * UnitPrice, 2),
          PaymentMethod = tostring(payments[toint(rand() * 10)]), Channel = tostring(channels[toint(rand() * 3)])""")
    print(f"  {len(history):,} hourly readings are in, and the history's shoppers have reacted to them.")

print("The store is open." + (" The weather feed is on." if INCLUDE_WEATHER else "") + (f" Muted region: {MUTE_REGION}." if MUTE_REGION else ""))
sent = extra = 0
latest, last_sent, last_fetch, last_report, started = {}, {}, 0.0, 0.0, time.time()
while time.time() - started < RUN_MINUTES * 60:
    tick = time.time()
    note = ""
    if INCLUDE_WEATHER and tick - last_fetch >= 300:                # every 5 minutes: ask the weather service
        try:
            readings = weather_now()
            changed = [r for r in readings
                       if last_sent.get(r["City"], (None, 0))[0] != (r["TempC"], r["PrecipMm"], r["WeatherCode"], r["WindKmh"])
                       or tick - last_sent[r["City"]][1] > 870]     # send when the values changed, or every 15 minutes
            if changed:
                send(weather, changed)
                for r in changed:
                    last_sent[r["City"]] = ((r["TempC"], r["PrecipMm"], r["WeatherCode"], r["WindKmh"]), tick)
            latest = {r["City"]: r for r in readings}
            note = " | raining now: " + (", ".join(c for c, r in latest.items() if r["RainRate"] >= RAIN) or "nowhere")
        except Exception as problem:                                # a failed weather call must not stop the store
            note = " | weather call failed, will try again: " + str(problem)[:80]
        last_fetch = tick
    now = datetime.now(timezone.utc)
    busy = CURVE[now.hour] * (1.12 if now.weekday() >= 5 else 1.0)
    per_batch = ORDERS_PER_MINUTE * busy / 6                         # one batch every 10 seconds
    batch = [order() for _ in range(max(1, round(per_batch * random.uniform(0.6, 1.4))))]
    weather_orders = []
    for city, r in latest.items():                                  # the weather-driven extra orders
        city_share = REGIONS[CITIES[city][2]][1] / TOTAL_SHARE / len(CITY_NAMES[CITIES[city][2]])
        for tag in what_weather_does(r):
            names, times = EFFECTS[tag]
            for product in (p for p in PRODUCTS if p[1] in names):
                weather_orders += [order(product, city) for _ in range(how_many(per_batch * city_share * product[4] / TOTAL_POPULAR * (times - 1)))]
    batch = [o for o in batch if o["Region"] != MUTE_REGION]
    weather_orders = [o for o in weather_orders if o["Region"] != MUTE_REGION]
    if batch or weather_orders:
        send(orders, batch + weather_orders)
        sent += len(batch)
        extra += len(weather_orders)
    if note or tick - last_report >= 60:
        print(f"{now:%H:%M:%S} UTC  orders sent: {sent}" + (f" + {extra} because of the weather" if INCLUDE_WEATHER else "") + note)
        last_report = tick
    time.sleep(max(0.0, 10 - (time.time() - tick)))
print("The store is closed. Orders sent:", sent + extra)